# Synthetic data from knowledge

References:

* [Nemotron-CC](https://arxiv.org/abs/2412.02595v2)
* [BeyondWeb](https://blog.datologyai.com/beyondweb/)
* [Kimi K2](https://arxiv.org/abs/2507.20534)

In [5]:
from pathlib import Path

OUTPUT_DIR = Path("../output/synthetic/v2")
INPUT_DIR = Path("../output/wiki/fandom/crawl")

## Rephraser model

In [ ]:
import dspy
from pydantic import BaseModel


class QuestionAnswerPair(BaseModel):
    """A question and its corresponding answer."""
    question: str
    answer: str


class QAGenerator(dspy.Signature):
    """Read the text, ask questions and answer them.

Follow these instructions:
1. Ask diverse questions that require different cognitive skills or cover different aspects of the text.
2. Ask questions in various forms such as:
- Yes/No questions that require determining whether a statement is true or false.
- Open-ended questions that begin with words like what, how, when, where, why and who.
- Multi-choice questions that offers two or more options to choose from. Include the options in the question.
- Comparison questions that compare two quantities or objects and determine the relationship between them.
- Reading comprehension questions that test the ability to understand and analyze the text.
- Problem-solving questions that test the ability to solve mathematical, physical, or logical problems.
- Long-form questions that require detailed answers or explanations.
3. Use the document summary and previous question-answer pairs as context.
- Do not repeat questions that have already been asked, or that are too similar to previously asked questions.
- New questions should be relevant only to the current document segment.
- New questions should follow naturally from the previous questions and answers.
3. Focus on asking questions about factual information, important knowledge, or concrete details in the text.
4. Write questions and answers using clear and concise language.
5. Use plain text. Do not use Markdown."""

    summary: str = dspy.InputField(
        desc="A brief summary of the whole document, for context."
    )
    previous_qa_pairs: list[QuestionAnswerPair] = dspy.InputField(
        desc="A list of previously asked question-answer pairs for the previous segment of the document."
    )
    document_segment: str = dspy.InputField(
        desc="The document segment to ask questions about."
    )
    question_answers: list[QuestionAnswerPair] = dspy.OutputField(
        desc="A list of question-answer pairs.",
        max_length=8,
    )


class SyntheticQAGenerator(dspy.Module):
    """Generate synthetic question-answer pairs from text segments."""

    def __init__(self):
        self.summarizer = dspy.Predict(dspy.make_signature(
            signature="text -> summary",
            instructions="Summarize the following text in a concise manner.",
        ))
        self.qa_generator = dspy.ChainOfThought(QAGenerator)

    def forward(self, doc: str, segments: list[str]) -> dspy.Prediction:
        summary = self.summarizer(text=doc).summary

        res = []
        prev = []
        for segment in segments:
            qa_pairs = self.qa_generator(
                summary=summary,
                previous_qa_pairs=prev,
                document_segment=segment
            ).question_answers
            res.extend(qa_pairs)
            prev = qa_pairs

        return dspy.Prediction(summary=summary, qa_pairs=res)

## Chunking

In [6]:
from markdown_it import MarkdownIt


def split_markdown(md_text: str) -> list[str]:
    md = MarkdownIt()
    tokens = md.parse(md_text)

    chunks = []
    current = []
    for i, tok in enumerate(tokens):
        if tok.type == "heading_open":
            # flush previous chunk
            if current:
                chunks.append("".join(current))
                current = []
        # reconstruct token text
        if tok.type == "inline":
            current.append(tok.content + "\n")
        elif tok.type.endswith("_open") or tok.type.endswith("_close"):
            # keep markup like ###, code fences, list markers
            current.append(tok.markup + "\n")
        elif tok.type == "fence":
            current.append(f"{tok.markup}{tok.info}\n{tok.content}{tok.markup}\n")
        elif tok.content:
            current.append(tok.content + "\n")

    if current:
        chunks.append("".join(current))

    return chunks

In [7]:
with open("../output/wiki/issues.md") as f:
    issues_md = f.read()

chunks = split_markdown(issues_md)

In [27]:
with open("../output/wiki/uno.md") as f:
    uno_md = f.read()

tokens = MarkdownIt().parse(uno_md)
set(token.type for token in tokens)

{'bullet_list_close',
 'bullet_list_open',
 'heading_close',
 'heading_open',
 'inline',
 'list_item_close',
 'list_item_open',
 'paragraph_close',
 'paragraph_open'}

In [66]:
[
    # (token.content if token.content else token.markup, token.type)
    token.content if token.content else token.markup
    for token in tokens
    if not token.type.endswith("_close")
]

def tokens_to_markdown(tokens):
    out = []
    in_list = False
    for tok in tokens:
        if tok.type == "inline":
            out.append(tok.content)
        elif tok.type == "fence":
            out.append(f"{tok.markup}{tok.info}\n{tok.content}{tok.markup}\n")
        elif tok.type == "code_block":
            out.append(tok.content + "\n")
        elif tok.type.endswith("_open"):
            if tok.tag.startswith("h"):  # heading
                level = int(tok.tag[1])
                # heading_open is followed by inline token
                # actual text handled by inline
                out.append("#" * level + " ")
            elif tok.type == "paragraph_open":
                # nothing, paragraph is implicit
                pass
            elif tok.type == "bullet_list_open":
                in_list = True
            elif tok.type == "list_item_open":
                out.append("- ")
        elif tok.type.endswith("_close"):
            if tok.tag.startswith("h"):
                out.append("\n\n")
            elif tok.type == "paragraph_close":
                if in_list:
                    out.append("\n")
                else:
                    out.append("\n\n")
            # elif tok.type == "list_item_close":
            #     out.append("\n")
            elif tok.type == "bullet_list_close":
                in_list = False
                out.append("\n")
        else:
            # fallback
            if tok.content:
                out.append(tok.content)
    return "".join(out)

print(tokens_to_markdown(tokens))

# Uno (personaggio)

Uno è uno dei personaggi principali della serie a fumetti Disney Italia di PK (PK - Paperinik New Adventures, PK², PK - Pikappa e PK - Nuova era).

## Biografia

Si tratta di un'intelligenza artificiale creata da Everett Ducklair. È in grado di controllare in ogni sua parte la Ducklair Tower, che ne costituisce il corpo centrale (tanto da presentarsi a Paperinik dicendo di essere a tutti gli effetti il palazzo), di svolgere calcoli di estrema difficoltà e di introdursi grazie alle sue superiori capacità nelle reti informatiche convenzionali più protette. Controlla anche la grande parte dei marchingegni costruiti da Ducklair.

È il più valido aiuto di Paperinik, di cui conosce l'identità segreta. L'aspetto con cui si presenta agli umani è molto simile al viso di Everett Ducklair, solitamente presentato in forma di ologramma all'interno di una sfera verde. Il suo acerrimo nemico è Due, la crudele intelligenza artificiale di back-up, creata da Everett per subentrargli

In [ ]:
from dataclasses import dataclass, field


def join_titles(titles: list[str]) -> str:
    return "\n".join(
        "#" * (i+1) + " " + title
        for i, title in enumerate(titles)
        if title.strip()
    ) + "\n"


@dataclass
class Chunk:
    headers: list[str] = field(default_factory=list)
    text: str = ""


def update_headers(headers: list[str], curr: str, level: str) -> list[str]:
    headers = headers.copy()
    level_num = int(level[1:]) - 1
    if level_num < len(headers):
        headers = headers[:level_num]
    headers.extend([""] * (level_num - len(headers)))
    headers.append(curr)
    return headers


# Idea: keep adding markdown to current chunk while we don't surpass chunk limits.
# Keep updating the current titles.
# To reconstruct context, look at two consecutive chunks and set the previous as context,
# while removing common headers from the current chunk.
def chunk_markdown(
        text: str,
        max_len: int = 1024,
        min_len: int = 256,
) -> list[Chunk]:
    headers: list[str] = []
    chunks: list[Chunk] = []
    curr = Chunk()

    tokens = MarkdownIt().parse(text)
    for token in tokens:
        pass
    return chunks


# Tests
assert update_headers(["one", "two"], "three", "h3") == ["one", "two", "three"]
assert update_headers(["one", "two"], "three", "h2") == ["one", "three"]
assert update_headers(["one", "two"], "three", "h1") == ["three"]
assert update_headers([], "three", "h1") == ["three"]
assert update_headers(["one"], "three", "h3") == ["one", "", "three"]